# RAG with ChromaDB: Indexing, Retrieval, Optimisation & LLM Connection

ChromaDB is a purpose-built, open-source vector database that stores embeddings and metadata in a persistent local directory (SQLite + Parquet). It requires zero infrastructure — just `pip install chromadb`.

This notebook covers:
- Loading and chunking a PDF document
- Building persistent ChromaDB collections
- Dual-store design (detail + summary collections for token optimisation)
- Retrieval strategies (similarity, MMR, metadata filtering, hybrid reranking)
- Index optimisation (HNSW parameters, distance functions, batch sizing)
- Connecting an LLM with RAG to reduce context token usage

In [ ]:
from __future__ import annotations

import os
import shutil
import re
import time
from pathlib import Path

import numpy as np
import chromadb
from chromadb.config import Settings as ChromaSettings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate

print("All imports OK")

## 1. Document Loading and Chunking

Same source document (`doc/NVDA_report.pdf`). We use overlapping chunks so sentence boundaries near a split are still captured.

In [ ]:
def find_rag_directory() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        direct = candidate / 'doc' / 'NVDA_report.pdf'
        nested = candidate / 'ai_tutorials' / 'rag' / 'doc' / 'NVDA_report.pdf'
        if direct.exists():
            return candidate
        if nested.exists():
            return candidate / 'ai_tutorials' / 'rag'
    raise FileNotFoundError('Could not find NVDA_report.pdf')

RAG_DIR = find_rag_directory()
DOC_PATH = RAG_DIR / 'doc' / 'NVDA_report.pdf'
VECTOR_DB_DIR = RAG_DIR / 'vector_db_chromadb'

loader = PyPDFLoader(str(DOC_PATH))
pages = loader.load()
print(f"Loaded {len(pages)} pages")

splitter = RecursiveCharacterTextSplitter(chunk_size=900, chunk_overlap=150)
chunks = splitter.split_documents(pages)
for i, c in enumerate(chunks):
    c.metadata['chunk_index'] = i
print(f"Created {len(chunks)} chunks")

## 2. Embedding Model

ChromaDB does not embed text itself — it relies on an external embedding function. We use the same Ollama model as the other notebooks.

In [ ]:
embeddings = OllamaEmbeddings(
    model="nomic-embed-text",
    base_url="http://localhost:11434",
)

test_vec = embeddings.embed_query("verify dimension")
print(f"Embedding dimension: {len(test_vec)}")

## 3. Building Persistent ChromaDB Collections

We build **two collections** to demonstrate a dual-store optimisation strategy:

1. **Detail collection**: one embedding per chunk — full retrieval coverage.
2. **Summary collection**: one compact metadata summary — cheap document-level context priming.

At query time, the summary is retrieved first (fast, ~1 vector lookup) to give the LLM document-level framing, then the detail chunks provide the specific evidence.

In [ ]:
if VECTOR_DB_DIR.exists():
    shutil.rmtree(VECTOR_DB_DIR)
VECTOR_DB_DIR.mkdir(parents=True, exist_ok=True)

# Initialise ChromaDB client
chroma_client = chromadb.PersistentClient(
    path=str(VECTOR_DB_DIR),
    settings=ChromaSettings(anonymized_telemetry=False),
)

COLLECTION_BASE = "nvda_report"
DETAIL_COLLECTION = f"{COLLECTION_BASE}_detail"
SUMMARY_COLLECTION = f"{COLLECTION_BASE}_summary"

# --- 3a. Detail collection (full chunk-level index) ---
print("Building detail collection...")
detail_store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=DETAIL_COLLECTION,
    persist_directory=str(VECTOR_DB_DIR),
    client=chroma_client,
)
print(f"  Detail collection: {detail_store._collection.count()} vectors")

# --- 3b. Summary collection (document-level metadata) ---
print("\nBuilding summary collection...")
preview = ' '.join(p.page_content.strip().replace('\n', ' ') for p in pages[:1])[:1000]
summary_text = (
    f"Document: {DOC_PATH.name}\n"
    f"Source: {DOC_PATH}\n"
    f"Total pages: {len(pages)}\n"
    f"Total chunks: {len(chunks)}\n"
    f"Preview: {preview}"
)

summary_store = Chroma(
    collection_name=SUMMARY_COLLECTION,
    persist_directory=str(VECTOR_DB_DIR),
    embedding_function=embeddings,
    client=chroma_client,
)
summary_store.add_texts(
    texts=[summary_text],
    metadatas=[{
        "doc_id": DOC_PATH.name,
        "source": str(DOC_PATH),
        "pages": len(pages),
        "chunks": len(chunks),
        "record_type": "document_summary",
    }],
    ids=[f"{DOC_PATH.name}_summary"],
)
print(f"  Summary collection: {summary_store._collection.count()} vectors")
print(f"\nVector DB stored in: {VECTOR_DB_DIR}")

## 4. Loading Persisted Collections

ChromaDB saves state automatically. Reloading reads the SQLite catalogue and memory-maps the Parquet embeddings — no re-computation.

In [ ]:
reloaded_detail = Chroma(
    collection_name=DETAIL_COLLECTION,
    persist_directory=str(VECTOR_DB_DIR),
    embedding_function=embeddings,
)
reloaded_summary = Chroma(
    collection_name=SUMMARY_COLLECTION,
    persist_directory=str(VECTOR_DB_DIR),
    embedding_function=embeddings,
)

print(f"Reloaded detail: {reloaded_detail._collection.count()} vectors")
print(f"Reloaded summary: {reloaded_summary._collection.count()} vectors")

## 5. Retrieval Strategies

ChromaDB supports several retrieval approaches. We demonstrate each with concrete Python code.

In [ ]:
question = "What does the report say about NVIDIA's data center revenue?"

# --- 5a. Plain similarity search ---
docs = reloaded_detail.similarity_search(question, k=4)
print("=== 5a. Plain similarity search ===")
for i, d in enumerate(docs):
    print(f"  [{i+1}] page={d.metadata.get('page')} chunk={d.metadata.get('chunk_index')} | {d.page_content[:80]}...")

# --- 5b. With distance scores (L2 distance) ---
docs_scored = reloaded_detail.similarity_search_with_score(question, k=4)
print("\n=== 5b. With distance scores ===")
for i, (d, score) in enumerate(docs_scored):
    print(f"  [{i+1}] score={score:.4f} | page={d.metadata.get('page')} | {d.page_content[:80]}...")

# --- 5c. MMR (Max Marginal Relevance) for diversity ---
docs_mmr = reloaded_detail.max_marginal_relevance_search(question, k=4, fetch_k=20, lambda_mult=0.7)
print("\n=== 5c. MMR (diversity) ===")
for i, d in enumerate(docs_mmr):
    print(f"  [{i+1}] page={d.metadata.get('page')} chunk={d.metadata.get('chunk_index')} | {d.page_content[:80]}...")

# --- 5d. Metadata filter (retrieve only specific pages) ---
filtered = reloaded_detail.similarity_search(
    question, k=4,
    filter={"page": {"$in": [1, 2]}},  # only pages 1 and 2 (0-indexed)
)
print("\n=== 5d. With metadata filter (pages 1-2) ===")
for i, d in enumerate(filtered):
    print(f"  [{i+1}] page={d.metadata.get('page')} chunk={d.metadata.get('chunk_index')} | {d.page_content[:80]}...")

# --- 5e. Hybrid: retrieve broadly + rerank with lexical overlap ---
def normalize_tokens(text: str) -> set[str]:
    return set(re.findall(r'[a-z0-9]+', text.lower()))

def hybrid_retrieve(store, question: str, k: int = 4, broad_k: int = 12):
    broad = store.similarity_search_with_score(question, k=broad_k)
    q_tokens = normalize_tokens(question)
    scored = []
    for doc, distance in broad:
        dense_score = 1.0 / (1.0 + float(distance))
        overlap = len(q_tokens & normalize_tokens(doc.page_content))
        fused = (0.75 * dense_score) + (0.25 * overlap)
        scored.append((fused, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[:k]

hybrid = hybrid_retrieve(reloaded_detail, question)
print("\n=== 5e. Hybrid (dense + lexical rerank) ===")
for i, (score, d) in enumerate(hybrid):
    print(f"  [{i+1}] fused={score:.4f} | page={d.metadata.get('page')} | {d.page_content[:80]}...")

## 6. Index Optimisation

ChromaDB uses HNSW by default. Key knobs:

| Parameter | Effect | Default | Tuning direction |
|-----------|--------|---------|------------------|
| `ef_construction` | Build quality vs. time | 100 | Higher = better recall, slower build |
| `ef_search` | Search quality vs. latency | 10 | Higher = better recall, slower search |
| `M` | Neighbours per graph node | 16 | Higher = better recall, more memory |
| `distance` | Distance function | "l2" | "cosine" for semantic similarity |

We benchmark different configurations below.

In [ ]:
test_queries = [
    "What is NVIDIA's revenue growth?",
    "Data center business performance",
    "Gaming segment highlights",
    "Forward guidance and outlook",
    "Stock buyback and dividends",
]

def benchmark_chromadb(name, store, queries):
    times = []
    for q in queries:
        start = time.perf_counter()
        _ = store.similarity_search_with_score(q, k=4)
        elapsed = time.perf_counter() - start
        times.append(elapsed)
    avg = np.mean(times) * 1000
    count = store._collection.count()
    print(f"  {name:30s}  avg {avg:6.1f} ms  vectors={count}")

print("Benchmarking ChromaDB (5 queries each):")
benchmark_chromadb("Default HNSW (ef=10)", reloaded_detail, test_queries)

# --- Build a collection with optimised HNSW ---
# ChromaDB uses a metadata dict to control HNSW parameters
optimised_dir = VECTOR_DB_DIR / "optimised"
optimised_client = chromadb.PersistentClient(
    path=str(optimised_dir),
    settings=ChromaSettings(anonymized_telemetry=False),
)

# Create collection with custom HNSW settings
optimised_collection = optimised_client.create_collection(
    name="optimised_hnsw",
    metadata={
        "hnsw:space": "cosine",            # cosine distance (0 = identical, 2 = opposite)
        "hnsw:construction_ef": 200,        # higher = better recall
        "hnsw:M": 32,                        # higher = better recall, more memory
        "hnsw:search_ef": 100,               # higher = better recall, slower
    },
)

# Embed all chunks and add to optimised collection
all_texts = [c.page_content for c in chunks]
all_metadatas = [c.metadata for c in chunks]
all_ids = [f"chunk_{i}" for i in range(len(chunks))]

print("\nBuilding optimised collection (cosine, M=32, ef=200/100)...")
embeddings_list = embeddings.embed_documents(all_texts)
optimised_collection.add(
    embeddings=embeddings_list,
    documents=all_texts,
    metadatas=all_metadatas,
    ids=all_ids,
)

optimised_store = Chroma(
    client=optimised_client,
    collection_name="optimised_hnsw",
    embedding_function=embeddings,
)
benchmark_chromadb("Optimised (cosine, M=32, ef=100)", optimised_store, test_queries)

# --- Compare with L2 distance ---
l2_dir = VECTOR_DB_DIR / "l2"
l2_client = chromadb.PersistentClient(
    path=str(l2_dir),
    settings=ChromaSettings(anonymized_telemetry=False),
)
l2_collection = l2_client.create_collection(
    name="l2_hnsw",
    metadata={
        "hnsw:space": "l2",
        "hnsw:construction_ef": 200,
        "hnsw:M": 32,
        "hnsw:search_ef": 100,
    },
)
l2_collection.add(
    embeddings=embeddings_list,
    documents=all_texts,
    metadatas=all_metadatas,
    ids=all_ids,
)
l2_store = Chroma(
    client=l2_client,
    collection_name="l2_hnsw",
    embedding_function=embeddings,
)
benchmark_chromadb("Optimised (L2, M=32, ef=100)", l2_store, test_queries)

## 7. Connecting an LLM with RAG

We build an optimised RAG pipeline that:
1. Retrieves document-level summary (fast, cheap)
2. Retrieves detail chunks broadly
3. Reranks with hybrid scoring
4. Fits everything under a token budget before calling the LLM

In [ ]:
chat = ChatOllama(
    model="llama3.1",
    base_url="http://localhost:11434",
    temperature=0,
)

prompt = ChatPromptTemplate.from_messages([
    ('system', 'You are a financial analyst. Answer the question using ONLY the context provided. '
               'If the context is insufficient, say so. Be concise.'),
    ('human', 'Question:\n{question}\n\nContext:\n{context}'),
])

def chroma_rag_answer(
    question: str,
    detail_store: Chroma,
    summary_store: Chroma,
    top_k: int = 4,
    token_budget: int = 800,
) -> tuple[str, dict]:
    """
    RAG with ChromaDB dual-store optimisation.
    """
    diagnostics = {}
    
    # Step 1: retrieve document summary (single vector lookup, very fast)
    summary_docs = summary_store.similarity_search(question, k=1)
    doc_summary = summary_docs[0].page_content if summary_docs else ""
    
    # Step 2: retrieve detail chunks broadly
    broad = detail_store.similarity_search_with_score(question, k=top_k * 3)
    diagnostics['broad_retrieved'] = len(broad)
    
    # Step 3: rerank with hybrid scoring
    q_tokens = normalize_tokens(question)
    scored = []
    for doc, distance in broad:
        dense_score = 1.0 / (1.0 + float(distance))
        overlap = len(q_tokens & normalize_tokens(doc.page_content))
        fused = (0.75 * dense_score) + (0.25 * overlap)
        scored.append((fused, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    selected = [doc for _, doc in scored[:top_k]]
    diagnostics['selected_after_rerank'] = len(selected)
    
    # Step 4: build context under token budget
    context_parts = [f"Document metadata:\n{doc_summary}"]
    token_count = len(doc_summary.split())
    for doc in selected:
        chunk_tokens = len(doc.page_content.split())
        if token_count + chunk_tokens > token_budget:
            continue
        context_parts.append(doc.page_content)
        token_count += chunk_tokens
    diagnostics['context_token_estimate'] = token_count
    diagnostics['token_budget'] = token_budget
    final_context = '\n\n---\n\n'.join(context_parts)
    
    # Step 5: call LLM
    chain = prompt | chat
    response = chain.invoke({'question': question, 'context': final_context})
    answer = response.content if hasattr(response, 'content') else str(response)
    
    return answer, diagnostics


question = "Summarise NVIDIA's data center revenue performance and outlook."
answer, diag = chroma_rag_answer(question, reloaded_detail, reloaded_summary)

print("=== Diagnostics ===")
for k, v in diag.items():
    print(f"  {k}: {v}")
print("\n=== Answer ===")
print(answer)

## 8. How RAG Optimises LLM Context

The dual-store design adds a document-level summary that the LLM sees for free (1 vector lookup). This frames the answer before the detailed evidence is presented.

| Strategy | Tokens to LLM | Cost / Query | Notes |
|----------|---------------|--------------|-------|
| Full document | ~15k | $0.0023 | Expensive, includes irrelevant pages |
| RAG (similarity only) | ~800 | $0.00012 | Good, but may miss exact terms |
| RAG + hybrid rerank | ~800 | $0.00012 | Better precision |
| RAG + dual-store | ~850 | $0.00013 | Best framing, small overhead |

The cost savings are **>90%** compared to sending the full document.

In [ ]:
full_text = ' '.join(c.page_content for c in chunks)
full_tokens = len(full_text.split())
rag_tokens = 850  # summary + detail chunks

print(f"Full document tokens: ~{full_tokens}")
print(f"RAG context tokens:   ~{rag_tokens}")
print(f"Reduction:            ~{100 * (1 - rag_tokens / full_tokens):.0f}%")
print()
print(f"Cost @ $0.15/M tokens:")
print(f"  Full doc:  ${full_tokens / 1_000_000 * 0.15:.6f}")
print(f"  With RAG:  ${rag_tokens / 1_000_000 * 0.15:.6f}")
print(f"  Savings:   {100 * (1 - rag_tokens / full_tokens):.0f}%")

## Summary

- **ChromaDB** provides zero-configuration persistent vector storage with SQLite + Parquet.
- **Dual-store** design (detail + summary collections) optimises the context/retrieval trade-off.
- **Retrieval strategies**: plain similarity, with scores, MMR, metadata filters, hybrid reranking.
- **Optimisation**: tune HNSW parameters (`ef_construction`, `M`, `ef_search`, distance function) via collection metadata.
- **RAG with LLM**: reduces context tokens by >90%, improves precision via hybrid reranking, and provides document-level framing via the summary store.